# 05. Dashboard Consolidado de Risco Municipal

**Objetivo:** Consolidar todas as análises preditivas em um dashboard unificado de risco municipal para tomada de decisão estratégica.

**Impacto no Negócio:** Fornecer visão integrada de riscos e oportunidades para alocação eficiente de recursos e priorização de ações.

**Dados de Entrada:** 
- `data/03_gold/ranking_risco_desmatamento_2023.parquet`
- `data/03_gold/ranking_risco_embargos_2023.parquet`
- `data/03_gold/tendencias_desmatamento_temporal.parquet`

**Dados de Saída:** `data/03_gold/dashboard_preditivo_consolidado.parquet`

In [ ]:
# ============================================================================
# MONTAR GOOGLE DRIVE (APENAS COLAB)
# ============================================================================

def montar_google_drive():
    """Monta o Google Drive no Colab."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive montado em /content/drive")
        return True
    except Exception as e:
        print(f"⚠️  Erro ao montar Google Drive: {e}")
        return False

# Detecta se está no Colab e tenta montar o Drive
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    print("Montando Google Drive...")
    montar_google_drive()
except ImportError:
    print("✓ Ambiente local detectado - não é necessário montar Drive")

In [ ]:
# ============================================================================
# CONFIGURAÇÃO DE AMBIENTE
# ============================================================================

import sys
import os
from pathlib import Path

# Detectar ambiente e configurar caminho corretamente
try:
    import google.colab
    print("📤 Ambiente Google Colab detectado")
    # No Colab, usar o diretório do drive
    if os.path.exists('/content/drive/MyDrive/dados_analise'):
        os.chdir('/content/drive/MyDrive/dados_analise')
        print("✓ Diretório alterado para: /content/drive/MyDrive/dados_analise")
    else:
        print("⚠️  Diretório dados_analise não encontrado no Drive")
except ImportError:
    print("✓ Ambiente local detectado")
    # Local, usar diretório atual
    current_dir = Path.cwd()
    # Se estiver em notebooks_analise_preditiva, voltar para o root
    if 'notebooks_analise_preditiva' in str(current_dir):
        os.chdir(current_dir.parent)
        print(f"✓ Diretório alterado para: {current_dir.parent}")

print(f"✓ Diretório de trabalho atual: {os.getcwd()}")

# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Constantes específicas para dashboard
THRESHOLD_CRITICO = 0.7
THRESHOLD_ALTO = 0.5
THRESHOLD_MODERADO = 0.3
NIVEIS_RISCO = ['Baixo', 'Moderado', 'Alto', 'Crítico']

# Configuração de caminhos
if os.path.exists('/content/drive'):
    caminho_base = '/content/drive/MyDrive/dados_analise/data/03_gold/'
else:
    caminho_base = 'data/03_gold/'

CAMINHO_DESMATAMENTO = f'{caminho_base}ranking_risco_desmatamento_2023.parquet'
CAMINHO_EMBARGOS = f'{caminho_base}ranking_risco_embargos_2023.parquet'
CAMINHO_TENDENCIAS = f'{caminho_base}tendencias_desmatamento_temporal.parquet'

if os.path.exists('/content/drive'):
    CAMINHO_SAIDA = '/content/data/03_gold/dashboard_preditivo_consolidado.parquet'
else:
    CAMINHO_SAIDA = 'data/03_gold/dashboard_preditivo_consolidado.parquet'

# Configuração do dashboard
PESO_DESMATAMENTO = 0.4
PESO_EMBARGOS = 0.4
PESO_TENDENCIA = 0.2

print(f"\nCaminho desmatamento: {CAMINHO_DESMATAMENTO}")
print(f"Caminho embargos: {CAMINHO_EMBARGOS}")
print(f"Caminho tendências: {CAMINHO_TENDENCIAS}")
print(f"Caminho saída: {CAMINHO_SAIDA}")

In [ ]:
## 1. Configuração e Importações
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURAÇÃO
# ============================================================================

# UFs da Amazônia Legal
UFS_AMAZONIA_LEGAL = ['AC', 'AM', 'AP', 'MA', 'MT', 'PA', 'RO', 'RR', 'TO']

In [ ]:
# ============================================================================
# PASSO 1: CARREGAMENTO DOS RANKINGS INDIVIDUAIS
# ============================================================================

# Carregar ranking de desmatamento usando função auxiliar
ranking_desmatamento = carregar_arquivo_parquet(CAMINHO_DESMATAMENTO)
print(f'Ranking de desmatamento: {len(ranking_desmatamento)} municípios')
print(f'Colunas: {ranking_desmatamento.columns.tolist()}')

# Carregar ranking de embargos usando função auxiliar
ranking_embargos = carregar_arquivo_parquet(CAMINHO_EMBARGOS)
print(f'\nRanking de embargos: {len(ranking_embargos)} municípios')
print(f'Colunas: {ranking_embargos.columns.tolist()}')

# Carregar tendências usando função auxiliar
tendencias = carregar_arquivo_parquet(CAMINHO_TENDENCIAS)
print(f'\nTendências temporais: {len(tendencias)} municípios')
print(f'Colunas: {tendencias.columns.tolist()}')

In [ ]:
# ============================================================================
# PASSO 2: CONSOLIDAÇÃO DOS DADOS
# ============================================================================

# Consolidar rankings usando função auxiliar
dashboard = consolidar_rankings(ranking_desmatamento, ranking_embargos, tendencias)

print(f'\nDashboard consolidado: {len(dashboard)} municípios')
print(f'Colunas: {dashboard.columns.tolist()}')

In [ ]:
# ============================================================================
# PASSO 3: NORMALIZAÇÃO DE TENDÊNCIAS
# ============================================================================

# Normalizar tendência para score 0-1 usando função auxiliar
dashboard = normalizar_tendencia(dashboard, 'tendencia_desmatamento')

print('Estatísticas de tendência normalizada:')
print(dashboard['tendencia_normalizada'].describe())

In [ ]:
# ============================================================================
# PASSO 4: CÁLCULO DE SCORE DE RISCO COMBINADO
# ============================================================================

# Calcular score de risco combinado usando função auxiliar
dashboard = calcular_score_risco_combinado(
    dashboard,
    PESO_DESMATAMENTO,
    PESO_EMBARGOS,
    PESO_TENDENCIA
)

print('\nEstatísticas do score de risco combinado:')
print(dashboard['score_risco_combinado'].describe())

In [ ]:
# ============================================================================
# PASSO 5: CLASSIFICAÇÃO DE NÍVEIS DE RISCO
# ============================================================================

# Classificar nível de risco usando função auxiliar
dashboard = classificar_dataframe_risco(
    dashboard,
    'score_risco_combinado',
    THRESHOLD_MODERADO,
    THRESHOLD_ALTO,
    THRESHOLD_CRITICO
)

print('\nDistribuição de níveis de risco:')
print(dashboard['nivel_risco'].value_counts().sort_index())

print('\nPercentual por nível:')
print(dashboard['nivel_risco'].value_counts(normalize=True).sort_index() * 100)

In [ ]:
# ============================================================================
# PASSO 6: ADIÇÃO DE RECOMENDAÇÕES
# ============================================================================

# Adicionar recomendações automáticas usando função auxiliar
dashboard = adicionar_recomendacoes(dashboard, 'nivel_risco')

print('\n=== RECOMENDAÇÕES POR NÍVEL DE RISCO ===')
for nivel in NIVEIS_RISCO:
    df_nivel = dashboard[dashboard['nivel_risco'] == nivel]
    if len(df_nivel) > 0:
        print(f'\n{nivel.upper()} ({len(df_nivel)} municípios):')
        print(f'  {df_nivel["recomendacao"].iloc[0]}')
        if len(df_nivel) <= 10:
            print(f'  Municípios: {", ".join(df_nivel["municipio"].tolist())}')
        else:
            print(f'  Municípios: {", ".join(df_nivel["municipio"].head(5).tolist())}...')

In [ ]:
# ============================================================================
# PASSO 7: TOP MUNICÍPIOS POR SCORE DE RISCO
# ============================================================================

# Top municípios por score de risco combinado
top_risco = dashboard.sort_values('score_risco_combinado', ascending=False).head(20)

print('\n=== TOP 20 MUNICÍPIOS - SCORE DE RISCO COMBINADO ===')
print(top_risco[['municipio', 'uf', 'score_risco_combinado', 'nivel_risco', 
                 'probabilidade_desmatamento_2023', 'probabilidade_embargos_2023',
                 'tendencia_desmatamento']].to_string(index=False))

In [ ]:
# ============================================================================
# PASSO 8: SALVAMENTO DO DASHBOARD
# ============================================================================

# Salvar dashboard consolidado usando função auxiliar
salvar_dashboard(dashboard, CAMINHO_SAIDA)

print(f'\nDashboard consolidado salvo em {CAMINHO_SAIDA}')
print(f'Total de municípios: {len(dashboard)}')
print(f'\nResumo da distribuição de risco:')
print(dashboard['nivel_risco'].value_counts().sort_index())

## 6. Classificação de Níveis de Risco

In [ ]:
# Classificar nível de risco
dashboard['nivel_risco'] = pd.cut(
    dashboard['score_risco_combinado'],
    bins=[0, THRESHOLD_MODERADO, THRESHOLD_ALTO, THRESHOLD_CRITICO, 1.0],
    labels=['Baixo', 'Moderado', 'Alto', 'Crítico']
)

print('\nDistribuição de níveis de risco:')
print(dashboard['nivel_risco'].value_counts().sort_index())

print('\nPercentual por nível:')
print(dashboard['nivel_risco'].value_counts(normalize=True).sort_index() * 100)

## 7. Análise por Nível de Risco

In [ ]:
# Análise detalhada por nível de risco
for nivel in ['Crítico', 'Alto', 'Moderado', 'Baixo']:
    df_nivel = dashboard[dashboard['nivel_risco'] == nivel]
    print(f'\n=== NÍVEL DE RISCO: {nivel.upper()} ===')
    print(f'Quantidade: {len(df_nivel)} municípios')
    if len(df_nivel) > 0:
        print(f'Score médio: {df_nivel["score_risco_combinado"].mean():.3f}')
        print(f'Prob. desmatamento média: {df_nivel["probabilidade_desmatamento_2023"].mean()*100:.1f}%')
        print(f'Prob. embargos média: {df_nivel["probabilidade_embargos_2023"].mean()*100:.1f}%')
        print(f'Tendência média: {df_nivel["tendencia_desmatamento"].mean():.1f} ha/ano')

## 8. Top Municípios por Score de Risco

In [ ]:
# Top municípios por score de risco combinado
top_risco = dashboard.sort_values('score_risco_combinado', ascending=False).head(20)

print('\n=== TOP 20 MUNICÍPIOS - SCORE DE RISCO COMBINADO ===')
print(top_risco[['municipio', 'uf', 'score_risco_combinado', 'nivel_risco', 
                 'probabilidade_desmatamento_2023', 'probabilidade_embargos_2023',
                 'tendencia_desmatamento']].to_string(index=False))

## 9. Recomendações Automáticas por Nível de Risco

In [ ]:
# Criar recomendações automáticas
def criar_recomendacao(nivel_risco):
    recomendacoes = {
        'Crítico': 'Ação imediata: Fiscalização intensiva, monitoramento satelital em tempo real, equipes dedicadas',
        'Alto': 'Ação recomendada: Fiscalização prioritária, alertas de monitoramento, visitas periódicas',
        'Moderado': 'Ação recomendada: Fiscalização regular, monitoramento intensificado, integração com sistemas de alerta',
        'Baixo': 'Ação recomendada: Monitoramento rotineiro, foco em prevenção, programas de educação ambiental'
    }
    return recomendacoes.get(nivel_risco, 'Sem recomendação')

dashboard['recomendacao'] = dashboard['nivel_risco'].apply(criar_recomendacao)

print('\n=== RECOMENDAÇÕES POR NÍVEL DE RISCO ===')
for nivel in ['Crítico', 'Alto', 'Moderado', 'Baixo']:
    df_nivel = dashboard[dashboard['nivel_risco'] == nivel]
    if len(df_nivel) > 0:
        print(f'\n{nivel.upper()} ({len(df_nivel)} municípios):')
        print(f'  {df_nivel["recomendacao"].iloc[0]}')
        if len(df_nivel) <= 10:
            print(f'  Municípios: {", ".join(df_nivel["municipio"].tolist())}')
        else:
            print(f'  Municípios: {", ".join(df_nivel["municipio"].head(5).tolist())}...')

## 10. Salvamento do Dashboard

In [ ]:
# Salvar dashboard consolidado
dashboard.to_parquet(CAMINHO_SAIDA, index=False)
print(f'\nDashboard consolidado salvo em {CAMINHO_SAIDA}')
print(f'Total de municípios: {len(dashboard)}')
print(f'\nResumo da distribuição de risco:')
print(dashboard['nivel_risco'].value_counts().sort_index())

## 11. Conclusão

**Resumo do Dashboard:**
- {len(dashboard)} municípios classificados por risco combinado
- Score de risco baseado em desmatamento ({PESO_DESMATAMENTO*100}%), embargos ({PESO_EMBARGOS*100}%) e tendências ({PESO_TENDENCIA*100}%)
- Distribuição: {len(dashboard[dashboard['nivel_risco']=='Crítico'])} críticos, {len(dashboard[dashboard['nivel_risco']=='Alto'])} altos, {len(dashboard[dashboard['nivel_risco']=='Moderado'])} moderados, {len(dashboard[dashboard['nivel_risco']=='Baixo'])} baixos

**Impacto no Negócio:**
- Visão integrada de riscos municipais
- Base para alocação eficiente de recursos
- Recomendações automáticas por nível de risco
- Priorização clara de ações

**Próximos Passos:**
- Implementar sistema de alertas automáticos
- Criar dashboard visual interativo
- Integrar com sistemas de monitoramento em tempo real
- Desenvolver API para acesso a previsões